# Volume arterial de referência do ImageCAS

Análise do volume físico das máscaras coronarianas (`label == 1`), na resolução nativa. Relaciona a distribuição dos volumes aos resultados canônicos Mid/High, sem executar o pipeline.

O cálculo utiliza o espaçamento de cada arquivo: $V_{\mathrm{mL}} = N_{\mathrm{voxels}}\,s_xs_ys_z/1000$, com espaçamentos em mm. Assim, 1 mL equivale a 1.000 mm³. O valor representa o espaço ocupado pela anotação, não fluxo sanguíneo.


## 1. Configuração

O CSV é salvo após cada exame. Nas próximas execuções, somente arquivos novos ou com erro são calculados. Use `FORCE_RECOMPUTE = True` se as máscaras concluídas forem alteradas. O diretório do dataset é resolvido por `IMAGECAS_BASE_PATH` ou pelos caminhos conhecidos do projeto.


In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import gaussian_kde
from IPython.display import display

# O bootstrap precede os imports compartilhados de utils.
REPO_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file() and (path / "src").is_dir()
)
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from utils.comparison_utils.artery_volume import (  # noqa: E402
    compare_artery_volume_cohorts,
    prepare_artery_volume_dice_data,
    prepare_predicted_artery_volumes,
    summarize_artery_volume_bands,
)
from utils.comparison_utils.io import load_split_results  # noqa: E402
from utils.project.analysis.artery_volume import (  # noqa: E402
    classify_artery_volume_outliers,
    load_or_compute_artery_volumes,
)
from utils.project.runtime.notebook_env import (  # noqa: E402
    configure_notebook_environment,
    get_default_split_paths,
)

REPO_ROOT = configure_notebook_environment(chdir_to_src=False)
CSV_PATH = (
    REPO_ROOT
    / "output"
    / "segmentation"
    / "analysis"
    / "imagecas_artery_volume"
    / "artery_volumes.csv"
)
FORCE_RECOMPUTE = False
PLOT_SPLIT = "test"  # train, val, test ou all (30 + 270 + 700 exames).
HISTOGRAM_BINS = 30
EXTREME_CASE_COUNT = 5
OUTLIER_METHOD = "std"  # Alternativas: "iqr" e "std".
IQR_MULTIPLIER = 1.5
STD_MULTIPLIER = 2.0  # 2.0

In [2]:
# Rótulos e estilo pertencem à apresentação; o pareamento fica no módulo.
OSTIA_COLORS = {"Óstios válidos": "#247A58", "Falha dos óstios": "#C44747"}
SPLIT_LABELS = {
    "train": "Treino",
    "val": "Validação",
    "test": "Teste",
    "all": "Conjunto completo",
}


def plot_volume_dice_panels(data, pairs, title, *, y_column="dice"):
    if data.empty:
        print("Nenhum exame disponível para esta comparação.")
        return None

    valid_ids = (
        pairs.loc[pairs["ostia_success_both"], "IMG_ID"] if not pairs.empty else []
    )
    # A coorte condicionada deve ter dados exibíveis nas duas resoluções.
    eligible_ids = set(data.loc[data["resolution"].eq("mid_res"), "IMG_ID"]) & set(
        data.loc[data["resolution"].eq("high_res"), "IMG_ID"]
    )
    valid_ids = [img_id for img_id in valid_ids if img_id in eligible_ids]
    axis_limit = (
        volume_plot_xlim
        if y_column == "dice"
        else (0, max(float(data[["volume_ml", y_column]].max().max()) * 1.05, 1.0))
    )
    cohorts = {
        "Todos": data,
        "Óstios válidos em ambas": data.loc[data["IMG_ID"].isin(valid_ids)],
    }
    fig = make_subplots(
        rows=1,
        cols=2,
        shared_xaxes="all" if y_column == "dice" else False,
        shared_yaxes=y_column == "dice",
        subplot_titles=("Mid", "High"),
        horizontal_spacing=0.07,
    )
    trace_cohorts = []
    annotations_by_cohort = {}
    for cohort_index, (cohort_name, cohort) in enumerate(cohorts.items()):
        annotations = []
        legend_seen = set()
        for col, (resolution, label) in enumerate(
            (("mid_res", "Mid"), ("high_res", "High")), start=1
        ):
            group = cohort.loc[cohort["resolution"].eq(resolution)]
            annotation = fig.layout.annotations[col - 1].to_plotly_json()
            annotation["text"] = f"{label} (N={len(group)})"
            annotations.append(annotation)
            for success, status_label in (
                (True, "Óstios válidos"),
                (False, "Falha dos óstios"),
            ):
                for outlier, symbol in ((False, "circle"), (True, "x")):
                    selected = group.loc[
                        group["ostia_success"].eq(success)
                        & group["is_volume_outlier"].eq(outlier)
                    ]
                    volume_label = "Outlier" if outlier else "Dentro da faixa"
                    legend_name = f"{status_label} · {volume_label}"
                    customdata = [
                        [
                            img_id,
                            label,
                            status_label,
                            {
                                "low": "Baixo (outlier)",
                                "in_range": "Dentro da faixa",
                                "high": "Alto (outlier)",
                            }[volume_group],
                            dice_value,
                        ]
                        for img_id, volume_group, dice_value in selected[
                            ["IMG_ID", "volume_outlier_group", "dice"]
                        ].itertuples(index=False, name=None)
                    ]
                    fig.add_trace(
                        go.Scatter(
                            x=selected["volume_ml"].tolist(),
                            y=selected[y_column].tolist(),
                            mode="markers",
                            name=legend_name,
                            legendgroup=legend_name,
                            showlegend=bool(len(selected))
                            and legend_name not in legend_seen,
                            visible=cohort_index == 0,
                            customdata=customdata,
                            marker=dict(
                                color=OSTIA_COLORS[status_label],
                                symbol=symbol,
                                size=7,
                                opacity=0.65,
                            ),
                            hovertemplate=(
                                "Exame: %{customdata[0]}<br>Resolução: %{customdata[1]}"
                                "<br>Referência: %{x:.3f} mL"
                                + (
                                    "<br>Dice: %{y:.4f}"
                                    if y_column == "dice"
                                    else "<br>Predição: %{y:.3f} mL<br>Dice: %{customdata[4]:.4f}"
                                )
                                + "<br>Óstios: %{customdata[2]}<br>Volume: %{customdata[3]}"
                                "<extra></extra>"
                            ),
                        ),
                        row=1,
                        col=col,
                    )
                    trace_cohorts.append(cohort_name)
                    if len(selected):
                        legend_seen.add(legend_name)
        if cohort.empty:
            annotations.append(
                dict(
                    text="Nenhum exame elegível nesta coorte.",
                    x=0.5,
                    y=0.5,
                    xref="paper",
                    yref="paper",
                    showarrow=False,
                )
            )
        annotations_by_cohort[cohort_name] = annotations

    buttons = [
        dict(
            label=cohort_name,
            method="update",
            args=[
                {"visible": [key == cohort_name for key in trace_cohorts]},
                {
                    "annotations": annotations_by_cohort[cohort_name],
                    "title": {"text": f"{title} — {cohort_name}"},
                },
            ],
        )
        for cohort_name in cohorts
    ]
    fig.update_xaxes(
        title_text="Volume arterial de referência (mL)",
        range=list(axis_limit),
        autorange=False,
    )
    fig.update_yaxes(
        range=[0, 1] if y_column == "dice" else list(axis_limit), autorange=False
    )
    fig.update_yaxes(
        title_text="Dice" if y_column == "dice" else "Volume predito (mL)", row=1, col=1
    )
    if y_column != "dice":
        for col in (1, 2):
            fig.add_shape(
                type="line",
                x0=0,
                y0=0,
                x1=axis_limit[1],
                y1=axis_limit[1],
                line=dict(color="#555555", dash="dash"),
                row=1,
                col=col,
            )
            fig.update_yaxes(
                scaleanchor="x" if col == 1 else "x2", scaleratio=1, row=1, col=col
            )
    fig.update_layout(
        title=dict(text=f"{title} — Todos"),
        annotations=annotations_by_cohort["Todos"],
        template="plotly_white",
        height=560,
        margin=dict(t=130, b=110),
        legend=dict(orientation="h", y=-0.25, x=0.5, xanchor="center"),
        updatemenus=[
            dict(
                buttons=buttons,
                direction="down",
                x=0,
                y=1.22,
                xanchor="left",
                yanchor="top",
                active=0,
            )
        ],
    )
    return fig


def plot_volume_histogram(data, limits, bins):
    if data.empty:
        print("Não há volumes válidos para o histograma.")
        return None
    values = data["volume_ml"].to_numpy(dtype=float)
    counts, edges = np.histogram(values, bins=bins)
    centers = (edges[:-1] + edges[1:]) / 2
    widths = np.diff(edges)
    fig = go.Figure(
        go.Bar(
            x=centers,
            y=counts,
            width=widths,
            name="Exames",
            marker_color="#4682B4",
            customdata=np.column_stack((edges[:-1], edges[1:])),
            hovertemplate="Intervalo: %{customdata[0]:.3f}–%{customdata[1]:.3f} mL"
            "<br>Exames: %{y}<extra></extra>",
        )
    )
    peak = float(counts.max())
    if len(values) > 1 and np.ptp(values) > 0:
        try:
            xs = np.linspace(edges[0], edges[-1], 250)
            ys = gaussian_kde(values)(xs) * len(values) * widths[0]
            fig.add_trace(
                go.Scatter(
                    x=xs,
                    y=ys,
                    mode="lines",
                    name="Densidade suavizada",
                    line=dict(color="#365E80"),
                )
            )
            peak = max(peak, float(ys.max()))
        except (np.linalg.LinAlgError, ValueError):
            print("Curva de densidade indisponível; contagens preservadas.")
    lines = [
        (float(np.mean(values)), "Média", "#B22222", "dash"),
        (float(np.median(values)), "Mediana", "#006400", "dot"),
    ]
    if limits["exam_count"]:
        lines.extend(
            [
                (limits["lower_limit_ml"], "Limite inferior", "#D97706", "dashdot"),
                (limits["upper_limit_ml"], "Limite superior", "#7E22CE", "dashdot"),
            ]
        )
    for value, label, color, dash in lines:
        if np.isfinite(value):
            fig.add_trace(
                go.Scatter(
                    x=[value, value],
                    y=[0, max(peak * 1.08, 1)],
                    mode="lines",
                    name=f"{label}: {value:.2f} mL",
                    line=dict(color=color, dash=dash),
                    hovertemplate=f"{label}: {value:.3f} mL<extra></extra>",
                )
            )
    fig.update_layout(
        title=f"Volume coronariano de referência — ImageCAS (N={len(values)})",
        template="plotly_white",
        height=500,
        barmode="overlay",
        xaxis_title="Volume arterial (mL)",
        yaxis_title="Número de exames",
        legend=dict(orientation="h", y=-0.22),
        margin=dict(b=120),
    )
    return fig


def plot_volume_dice_delta(pairs):
    if pairs.empty:
        print("Nenhum par Mid/High com volume conhecido.")
        return None
    cohorts = {
        "Todos": pairs,
        "Óstios válidos em ambas": pairs.loc[pairs["ostia_success_both"]],
    }
    fig = go.Figure()
    trace_cohorts = []
    for cohort_index, (name, cohort) in enumerate(cohorts.items()):
        for success, color in ((True, "#247A58"), (False, "#C44747")):
            status = (
                "Óstios válidos em ambas" if success else "Falha em alguma resolução"
            )
            for outlier, symbol in ((False, "circle"), (True, "x")):
                group = cohort.loc[
                    cohort["ostia_success_both"].eq(success)
                    & cohort["is_volume_outlier"].eq(outlier)
                ]
                custom = [
                    [
                        row.IMG_ID,
                        row.dice_mid,
                        row.dice_high,
                        "Válidos" if row.ostia_success_mid else "Falha",
                        "Válidos" if row.ostia_success_high else "Falha",
                        {
                            "low": "Baixo (outlier)",
                            "in_range": "Dentro da faixa",
                            "high": "Alto (outlier)",
                        }[row.volume_outlier_group],
                    ]
                    for row in group.itertuples(index=False)
                ]
                fig.add_trace(
                    go.Scatter(
                        x=group["volume_ml"].tolist(),
                        y=group["delta_dice"].tolist(),
                        customdata=custom,
                        mode="markers",
                        visible=cohort_index == 0,
                        name=f"{status} · {'Outlier' if outlier else 'Dentro da faixa'}",
                        marker=dict(color=color, symbol=symbol, size=7, opacity=0.65),
                        showlegend=bool(len(group)),
                        hovertemplate="Exame: %{customdata[0]}<br>Volume: %{x:.3f} mL"
                        "<br>Dice Mid: %{customdata[1]:.4f}<br>Dice High: %{customdata[2]:.4f}"
                        "<br>Δ Dice: %{y:.4f}<br>Óstios Mid: %{customdata[3]}"
                        "<br>Óstios High: %{customdata[4]}<br>Volume: %{customdata[5]}<extra></extra>",
                    )
                )
                trace_cohorts.append(name)
    title = f"{SPLIT_LABELS[PLOT_SPLIT]}: Δ Dice (High − Mid)"
    buttons = []
    for name, cohort in cohorts.items():
        buttons.append(
            dict(
                label=name,
                method="update",
                args=[
                    {"visible": [key == name for key in trace_cohorts]},
                    {
                        "title": {"text": f"{title} — {name} (N={len(cohort)})"},
                        "annotations": []
                        if len(cohort)
                        else [
                            dict(
                                text="Nenhum exame elegível nesta coorte.",
                                x=0.5,
                                y=0.5,
                                xref="paper",
                                yref="paper",
                                showarrow=False,
                            )
                        ],
                    },
                ],
            )
        )
    fig.add_hline(y=0, line_dash="dash", line_color="#555555")
    fig.update_layout(
        title=f"{title} — Todos (N={len(pairs)})",
        template="plotly_white",
        height=550,
        xaxis=dict(
            title="Volume arterial de referência (mL)",
            range=list(volume_plot_xlim),
            autorange=False,
        ),
        yaxis=dict(title="Δ Dice (High − Mid)", range=[-1, 1], autorange=False),
        updatemenus=[dict(buttons=buttons, x=0, y=1.2, active=0)],
        legend=dict(orientation="h", y=-0.22),
        margin=dict(t=120, b=120),
    )
    return fig


def plot_volume_band_performance(summary):
    if summary.empty:
        print("Sem dados para comparar faixas de volume.")
        return None
    fig = make_subplots(
        rows=1, cols=2, subplot_titles=("Dice médio ± DP", "Sucesso dos óstios")
    )
    for resolution, label, color in (
        ("mid_res", "Mid", "#3574A5"),
        ("high_res", "High", "#B76A32"),
    ):
        group = summary.loc[summary["resolution"].eq(resolution)].sort_values("band")
        custom = [
            [
                row.lower_ml,
                row.upper_ml,
                row.exam_count,
                row.dice_mean,
                "Indisponível (N=1)"
                if pd.isna(row.dice_std)
                else f"{row.dice_std:.4f}",
                row.dice_median,
                row.ostia_success_count,
            ]
            for row in group.itertuples(index=False)
        ]
        hover = (
            "Faixa: %{customdata[0]:.3f}–%{customdata[1]:.3f} mL"
            "<br>N: %{customdata[2]}<br>Dice médio: %{customdata[3]:.4f}"
            "<br>DP: %{customdata[4]}<br>Mediana: %{customdata[5]:.4f}"
            "<br>Sucessos: %{customdata[6]}<extra>%{fullData.name}</extra>"
        )
        for col, y in ((1, "dice_mean"), (2, "ostia_success_percent")):
            fig.add_trace(
                go.Bar(
                    x=[f"F{int(band)}" for band in group["band"]],
                    y=group[y].tolist(),
                    name=label,
                    legendgroup=label,
                    showlegend=col == 1,
                    marker_color=color,
                    offsetgroup=label,
                    customdata=custom,
                    hovertemplate=hover,
                    error_y=dict(
                        type="data", array=group["dice_std"].fillna(0).tolist()
                    )
                    if col == 1
                    else None,
                ),
                row=1,
                col=col,
            )
    fig.update_yaxes(title_text="Dice", range=[0, 1], row=1, col=1)
    fig.update_yaxes(title_text="Exames (%)", range=[0, 100], row=1, col=2)
    fig.update_xaxes(title_text="Faixa de volume de referência")
    fig.update_layout(
        title=f"{SPLIT_LABELS[PLOT_SPLIT]}: desempenho por faixa de volume",
        template="plotly_white",
        height=520,
        barmode="group",
    )
    return fig

## 2. Carregamento ou cálculo

O helper lê somente as máscaras, uma por vez, e converte as unidades espaciais NIfTI para mm (cabeçalhos sem unidade seguem a convenção mm do ImageCAS). Erros ficam registrados no CSV sem receber volume zero. Sem o dataset montado, o CSV existente continua consultável, com aviso de que novos arquivos não puderam ser verificados.


In [3]:
# Retoma o CSV sem reler as máscaras já concluídas.
results_df = load_or_compute_artery_volumes(
    CSV_PATH,
    force_recompute=FORCE_RECOMPUTE,
)
valid_df = results_df.loc[results_df["status"].eq("ok")].copy()
error_df = results_df.loc[results_df["status"].eq("error")].copy()
print(f"CSV: {CSV_PATH.relative_to(REPO_ROOT)}")

Volume arterial ImageCAS: 0it [00:00, ?it/s]

CSV: output/segmentation/analysis/imagecas_artery_volume/artery_volumes.csv


In [4]:
display(
    pd.DataFrame(
        {
            "Situação": ["Exames registrados", "Concluídos", "Com erro"],
            "Quantidade": [len(results_df), len(valid_df), len(error_df)],
        }
    )
)

,Situação,Quantidade
0,Exames registrados,1000
1,Concluídos,1000
2,Com erro,0


## 3. Estatísticas do volume em mL

Somente exames com status `ok` entram nas estatísticas. O desvio-padrão é amostral; 25%, 50% e 75% correspondem ao primeiro quartil, mediana e terceiro quartil. Um volume realmente vazio permanece zero; uma falha de leitura permanece ausente.


In [5]:
if valid_df.empty:
    print("Não há exames concluídos para calcular estatísticas.")
else:
    statistics = valid_df["volume_ml"].describe().drop("count")
    statistics.index = [
        "Média",
        "Desvio-padrão",
        "Mínimo",
        "Primeiro quartil",
        "Mediana",
        "Terceiro quartil",
        "Máximo",
    ]
    display(statistics.rename("Volume (mL)").to_frame().round(4))

,Volume (mL)
Média,6.9990
Desvio-padrão,2.1144
Mínimo,1.9211
Primeiro quartil,5.5036
Mediana,6.7376
Terceiro quartil,8.1177
Máximo,16.5240


## 4. Volumes baixos, altos e outliers

Por padrão, os limites são $Q_1-1{,}5\,IQR$ e $Q_3+1{,}5\,IQR$, com $IQR=Q_3-Q_1$. Valores exatamente nos limites não são outliers. A alternativa `OUTLIER_METHOD = "std"` usa média ± `STD_MULTIPLIER` desvios-padrão amostrais.

Os limites são descritivos, calculados sobre todos os volumes válidos do CSV. A distância padronizada é $z=(V-\bar V)/s$: valores negativos indicam volumes abaixo da média; positivos, acima. Um outlier volumétrico não implica falha ou anotação incorreta.


In [6]:
# A classificação é feita em memória; o CSV de volumes não é modificado.
classified_volumes_df, outlier_limits = classify_artery_volume_outliers(
    results_df,
    method=OUTLIER_METHOD,
    iqr_multiplier=IQR_MULTIPLIER,
    std_multiplier=STD_MULTIPLIER,
)
display(
    pd.DataFrame(
        {
            "Critério": [OUTLIER_METHOD.upper()],
            "Exames válidos": [outlier_limits["exam_count"]],
            "Limite inferior (mL)": [outlier_limits["lower_limit_ml"]],
            "Limite superior (mL)": [outlier_limits["upper_limit_ml"]],
        }
    ).round(4)
)

,Critério,Exames válidos,Limite inferior (mL),Limite superior (mL)
0,STD,1000,2.7702,11.2278


In [7]:
counts = classified_volumes_df["volume_outlier_group"].value_counts()
outlier_counts_df = pd.DataFrame(
    {
        "Grupo": ["Volume baixo (outlier)", "Dentro da faixa", "Volume alto (outlier)"],
        "Quantidade": [int(counts.get(key, 0)) for key in ("low", "in_range", "high")],
    }
)
outlier_counts_df["Percentual (%)"] = (
    outlier_counts_df["Quantidade"] / len(classified_volumes_df) * 100
    if len(classified_volumes_df)
    else float("nan")
)
display(outlier_counts_df.round(2))

,Grupo,Quantidade,Percentual (%)
0,Volume baixo (outlier),4,0.4
1,Dentro da faixa,952,95.2
2,Volume alto (outlier),44,4.4


In [8]:
outlier_exams_df = classified_volumes_df.loc[
    classified_volumes_df["is_volume_outlier"],
    ["IMG_ID", "volume_ml", "volume_outlier_group", "volume_z_score"],
].sort_values("volume_ml")
if outlier_exams_df.empty:
    print("Nenhum outlier identificado pelo critério selecionado.")
else:
    display(
        outlier_exams_df.replace(
            {"volume_outlier_group": {"low": "Baixo", "high": "Alto"}}
        )
        .rename(
            columns={
                "volume_ml": "Volume (mL)",
                "volume_outlier_group": "Grupo",
                "volume_z_score": "Distância da média (DP)",
            }
        )
        .reset_index(drop=True)
        .round(4)
    )

,IMG_ID,Volume (mL),Grupo,Distância da média (DP)
0,809,1.9211,Baixo,-2.4016
1,655,2.1303,Baixo,-2.3026
2,276,2.4051,Baixo,-2.1727
3,868,2.7479,Baixo,-2.0106
4,386,11.2558,Alto,2.0132
5,140,11.2768,Alto,2.0232
6,500,11.3961,Alto,2.0796
7,963,11.4134,Alto,2.0878
8,993,11.4443,Alto,2.1024
9,193,11.4461,Alto,2.1033


## 5. Distribuição dos volumes

Cada observação corresponde a um exame. A curva suavizada ajuda a visualizar a distribuição; as linhas verticais indicam média e mediana.


In [9]:
volume_histogram_figure = plot_volume_histogram(
    valid_df, outlier_limits, HISTOGRAM_BINS
)
if volume_histogram_figure is not None:
    volume_histogram_figure.show()

## 6. Exames com menores e maiores volumes

Os extremos são candidatos à inspeção, não erros automáticos: diferenças anatômicas e de extensão da anotação também alteram o volume.


In [10]:
if valid_df.empty:
    print("Não há exames concluídos para selecionar os menores volumes.")
else:
    display(
        valid_df.nsmallest(EXTREME_CASE_COUNT, "volume_ml")
        .loc[:, ["IMG_ID", "volume_ml", "voxel_count", "label_file"]]
        .rename(columns={"volume_ml": "Volume (mL)", "voxel_count": "Voxels"})
        .reset_index(drop=True)
    )

,IMG_ID,Volume (mL),Voxels,label_file
0,809,1.921140,37910,809.label.nii.gz
1,655,2.130277,35650,655.label.nii.gz
2,276,2.405111,38490,276.label.nii.gz
3,868,2.747852,44465,868.label.nii.gz
4,931,2.807508,39929,931.label.nii.gz


In [11]:
if valid_df.empty:
    print("Não há exames concluídos para selecionar os maiores volumes.")
else:
    display(
        valid_df.nlargest(EXTREME_CASE_COUNT, "volume_ml")
        .loc[:, ["IMG_ID", "volume_ml", "voxel_count", "label_file"]]
        .rename(columns={"volume_ml": "Volume (mL)", "voxel_count": "Voxels"})
        .reset_index(drop=True)
    )

,IMG_ID,Volume (mL),Voxels,label_file
0,307,16.523963,192758,307.label.nii.gz
1,347,15.577230,219253,347.label.nii.gz
2,293,14.968100,208513,293.label.nii.gz
3,39,14.929193,207971,39.label.nii.gz
4,677,14.904474,203411,677.label.nii.gz


## 7. Canonical Mid/High com e sem os outliers

A comparação usa somente as referências oficiais de `canonical`, por treino, validação, teste e total disponível. Os mesmos IDs classificados como outliers são excluídos em ambas as resoluções. Exames sem volume conhecido são informados e preservados; falhas documentadas sem Dice recebem zero.

Esta é uma análise descritiva de sensibilidade à composição da coorte. A exclusão usa o volume do ground truth e não representa melhoria validada do algoritmo nem substitui o resultado global. Nenhum resultado canônico é alterado.


In [12]:
# Carrega resultados salvos: não executa segmentações nem procura runs alternativos.
canonical_frames = {}
canonical_configs = {}
canonical_sources = []
comparison_df = pd.DataFrame()
if classified_volumes_df.empty:
    print("Não há volumes válidos para analisar a exclusão de outliers.")
else:
    split_paths = get_default_split_paths(REPO_ROOT)
    for resolution in ("mid_res", "high_res"):
        for split in ("train", "val", "test"):
            source = split_paths.get(resolution, {}).get(split)
            message = "Disponível"
            try:
                frame = load_split_results(split_paths, resolution, split)
            except FileNotFoundError as error:
                frame = None
                message = str(error)
            if frame is not None:
                canonical_frames[resolution, split] = frame
                config_path = (
                    source.parent if source.name == "numeric" else source
                ) / "config/effective_pipeline_config.json"
                if config_path.is_file():
                    with config_path.open() as handle:
                        canonical_configs[resolution, split] = json.load(handle)
                else:
                    message = "Resultados disponíveis; configuração ausente para volume predito"
            elif message == "Disponível":
                message = "Canonical ausente"
            canonical_sources.append(
                {
                    "Resolução": "Mid" if resolution == "mid_res" else "High",
                    "Conjunto": split,
                    "Exames": len(frame) if frame is not None else 0,
                    "Origem": str(source.relative_to(REPO_ROOT)) if source else "—",
                    "Situação": message,
                }
            )
    if canonical_frames:
        comparison_df = compare_artery_volume_cohorts(
            canonical_frames, classified_volumes_df
        )
        comparison_df["Resolução"] = comparison_df["resolution"].map(
            {"mid_res": "Mid", "high_res": "High"}
        )
        comparison_df["Conjunto"] = comparison_df["split"].map(
            {
                "train": "Treino",
                "val": "Validação",
                "test": "Teste",
                "all": "Todos disponíveis",
            }
        )
display(pd.DataFrame(canonical_sources))

,Resolução,Conjunto,Exames,Origem,Situação
0,Mid,train,30,output/segmentation/canonical/mid_res/train/cu...,Disponível
1,Mid,val,270,output/segmentation/canonical/mid_res/val/curr...,Disponível
2,Mid,test,700,output/segmentation/canonical/mid_res/test/cur...,Disponível
3,High,train,30,output/segmentation/canonical/high_res/train/c...,Disponível
4,High,val,270,output/segmentation/canonical/high_res/val/cur...,Disponível
5,High,test,700,output/segmentation/canonical/high_res/test/cu...,Disponível


In [13]:
if comparison_df.empty:
    print("Não há referências canônicas disponíveis para comparar.")
else:
    display(
        comparison_df[
            [
                "Resolução",
                "Conjunto",
                "total_exams",
                "excluded_exams",
                "remaining_exams",
                "dice_before",
                "dice_after",
                "delta_dice",
            ]
        ]
        .rename(
            columns={
                "total_exams": "N original",
                "excluded_exams": "Outliers excluídos",
                "remaining_exams": "N sem outliers",
                "dice_before": "Dice original",
                "dice_after": "Dice sem outliers",
                "delta_dice": "Δ Dice",
            }
        )
        .round(4)
    )

,Resolução,Conjunto,N original,Outliers excluídos,N sem outliers,Dice original,Dice sem outliers,Δ Dice
0,High,Treino,30,0,30,0.5720,0.5720,0.0000
1,Mid,Treino,30,0,30,0.6288,0.6288,0.0000
2,High,Validação,270,19,251,0.5155,0.5189,0.0034
3,Mid,Validação,270,19,251,0.6009,0.6045,0.0035
4,High,Teste,700,29,671,0.5146,0.5165,0.0019
5,Mid,Teste,700,29,671,0.5996,0.6011,0.0015
6,High,Todos disponíveis,1000,48,952,0.5165,0.5189,0.0023
7,Mid,Todos disponíveis,1000,48,952,0.6008,0.6029,0.0020


In [14]:
if not comparison_df.empty:
    display(
        comparison_df[
            [
                "Resolução",
                "Conjunto",
                "ostia_success_count_before",
                "ostia_success_count_after",
                "ostia_success_percent_before",
                "ostia_success_percent_after",
                "delta_ostia_success_pp",
            ]
        ]
        .rename(
            columns={
                "ostia_success_count_before": "Sucessos originais",
                "ostia_success_count_after": "Sucessos sem outliers",
                "ostia_success_percent_before": "Sucesso original (%)",
                "ostia_success_percent_after": "Sucesso sem outliers (%)",
                "delta_ostia_success_pp": "Δ sucesso (p.p.)",
            }
        )
        .round(2)
    )

,Resolução,Conjunto,Sucessos originais,Sucessos sem outliers,Sucesso original (%),Sucesso sem outliers (%),Δ sucesso (p.p.)
0,High,Treino,27,27,90.00,90.00,0.00
1,Mid,Treino,29,29,96.67,96.67,0.00
2,High,Validação,213,199,78.89,79.28,0.39
3,Mid,Validação,230,215,85.19,85.66,0.47
4,High,Teste,538,519,76.86,77.35,0.49
5,Mid,Teste,597,574,85.29,85.54,0.26
6,High,Todos disponíveis,778,745,77.80,78.26,0.46
7,Mid,Todos disponíveis,856,818,85.60,85.92,0.32


In [15]:
if not comparison_df.empty:
    missing_volume_df = comparison_df.loc[
        comparison_df["missing_volume_count"].gt(0),
        ["Resolução", "Conjunto", "missing_volume_count"],
    ]
    if missing_volume_df.empty:
        print("Todos os exames canônicos têm volume disponível no CSV.")
    else:
        print("Exames sem volume foram preservados, não excluídos automaticamente.")
        display(
            missing_volume_df.rename(columns={"missing_volume_count": "Sem volume"})
        )

Todos os exames canônicos têm volume disponível no CSV.


## 8. Volume arterial e desempenho canônico

Selecione `PLOT_SPLIT` na configuração (padrão: teste). Use `all` para reunir treino, validação e teste, totalizando 1.000 exames quando as três referências estiverem disponíveis. A tabela de cobertura identifica conjuntos ausentes; IDs duplicados entre conjuntos são rejeitados. O volume predito usa a configuração efetiva de cada origem, mesmo na seleção completa. O eixo de volume usa a máscara coronariana de referência em mL. Os outliers continuam incluídos; a análise é descritiva, não um ajuste do pipeline.

A tabela informa IDs sem volume ou sem resultado na outra resolução. A figura interativa alterna entre todos os exames com volume conhecido em cada resolução e os IDs comuns com ambos os óstios corretos ou toleráveis nas duas resoluções. O delta é calculado nos IDs comuns com volume, incluindo falhas documentadas.


In [16]:
volume_dice_df = pd.DataFrame()
volume_dice_pairs = pd.DataFrame()
if classified_volumes_df.empty:
    print("Sem volumes válidos para os gráficos.")
else:
    volume_dice_df, volume_dice_pairs, volume_dice_coverage = (
        prepare_artery_volume_dice_data(
            canonical_frames,
            classified_volumes_df,
            split=PLOT_SPLIT,
        )
    )
    display(
        volume_dice_coverage.rename(
            columns={
                "resolution": "Resolução",
                "split": "Conjunto",
                "source_available": "Canonical disponível",
                "missing_splits": "Conjuntos ausentes",
                "total_exams": "N canônico",
                "plotted_exams": "N com volume",
                "missing_volume_count": "Sem volume",
                "missing_volume_ids": "IDs sem volume",
                "missing_pair_count": "Sem par",
                "missing_pair_ids": "IDs sem par",
            }
        )
    )
volume_plot_xlim = (
    (0, max(float(volume_dice_df["volume_ml"].max()) * 1.05, 1.0))
    if not volume_dice_df.empty
    else (0, 1)
)

,Resolução,Conjunto,Canonical disponível,Conjuntos ausentes,N canônico,N com volume,Sem volume,IDs sem volume,Sem par,IDs sem par
0,mid_res,test,True,[],700,700,0,[],0,[]
1,high_res,test,True,[],700,700,0,[],0,[]


### 8.1 Volume × Dice: comparação interativa

O menu alterna entre todos os exames e a coorte comum com óstios válidos nas duas resoluções, mantendo os limites dos eixos. As cores distinguem sucesso e falha dos óstios; círculos e X distinguem volumes dentro da faixa e outliers. Passe o mouse para consultar ID, resolução, volume, Dice e classificações.

As falhas documentadas sem Dice são representadas por zero. O filtro condicionado não altera os resultados gerais nem exclui outliers automaticamente.


In [17]:
volume_dice_figure = plot_volume_dice_panels(
    volume_dice_df,
    volume_dice_pairs,
    f"{SPLIT_LABELS[PLOT_SPLIT]}: volume arterial × Dice",
)
if volume_dice_figure is not None:
    volume_dice_figure.show()

### 8.2 Volume × diferença de Dice entre resoluções

Cada ponto é um exame pareado. Valores positivos favorecem High; negativos favorecem Mid. A cor distingue sucesso dos óstios nas duas resoluções de falha em pelo menos uma delas. Os outliers são mantidos.


In [18]:
volume_delta_figure = plot_volume_dice_delta(volume_dice_pairs)
if volume_delta_figure is not None:
    volume_delta_figure.show()

### 8.3 Desempenho por faixa de volume

Os quartis são calculados com um volume por exame do conjunto selecionado; os limites são os mesmos em Mid e High. Limites repetidos reduzem a quantidade de faixas. Todos os exames com referência conhecida permanecem incluídos, inclusive falhas e outliers. As barras de erro mostram DP amostral, não intervalo de confiança; para N=1, o DP é indisponível.


In [19]:
volume_band_summary, volume_band_bounds = summarize_artery_volume_bands(volume_dice_df)
if not volume_band_bounds.empty:
    print(f"Faixas efetivas: {len(volume_band_bounds)} (até quatro quartis).")
    display(
        volume_band_bounds.rename(
            columns={
                "band": "Faixa",
                "lower_ml": "Limite inferior (mL)",
                "upper_ml": "Limite superior (mL)",
            }
        ).round(3)
    )

Faixas efetivas: 4 (até quatro quartis).


,Faixa,Limite inferior (mL),Limite superior (mL)
0,1,1.921,5.482
1,2,5.482,6.650
2,3,6.650,8.091
3,4,8.091,14.929


In [20]:
volume_band_figure = plot_volume_band_performance(volume_band_summary)
if volume_band_figure is not None:
    volume_band_figure.show()

### 8.4 Volume predito × volume de referência

O volume predito usa os voxels após a morfologia, multiplicados pelo volume físico do voxel na grade processada. Os espaçamentos nativos vêm do CSV de volumes e os fatores de redução vêm da configuração efetiva da mesma fonte canônica. Configurações, contagens ou geometrias ausentes não são substituídas por valores presumidos.

Na linha y=x os volumes são iguais; abaixo dela, o volume predito é menor; acima, maior. Isso não garante correspondência espacial nem diagnostica sozinho vazamento. Predições registradas com zero voxels são preservadas; valores ausentes são reportados e excluídos apenas deste gráfico. O menu condicionado exige dados exibíveis e óstios válidos nas duas resoluções.


In [21]:
predicted_volume_df = pd.DataFrame()
if not classified_volumes_df.empty:
    predicted_volume_df, predicted_volume_coverage = prepare_predicted_artery_volumes(
        canonical_frames,
        classified_volumes_df,
        canonical_configs,
        split=PLOT_SPLIT,
    )
    display(
        predicted_volume_coverage.rename(
            columns={
                "resolution": "Resolução",
                "total_exams": "N canônico",
                "missing_reference_count": "Sem volume de referência",
                "valid_predictions": "Volumes preditos válidos",
                "excluded_ids": "IDs excluídos",
                "status_counts": "Situação do cálculo",
            }
        )
    )

,Resolução,N canônico,Sem volume de referência,Volumes preditos válidos,IDs excluídos,Situação do cálculo
0,mid_res,700,0,700,[],{'ok': 700}
1,high_res,700,0,697,"[125, 86, 657]","{'ok': 697, 'missing_or_invalid_count': 3}"


In [22]:
predicted_volume_figure = None
if not predicted_volume_df.empty:
    predicted_volume_figure = plot_volume_dice_panels(
        predicted_volume_df.loc[
            predicted_volume_df["prediction_volume_status"].eq("ok")
        ],
        volume_dice_pairs,
        f"{SPLIT_LABELS[PLOT_SPLIT]}: volume predito × referência",
        y_column="predicted_volume_ml",
    )
if predicted_volume_figure is not None:
    predicted_volume_figure.show()

## 9. Erros e interpretação

O CSV preserva os erros para nova tentativa na execução seguinte. Os resultados descrevem a classe coronariana de referência, sem separar ramos esquerdo/direito e sem aplicar downsampling, threshold ou morfologia. Figuras e tabelas são exibidas aqui; apenas o CSV compacto é persistido.


In [23]:
if error_df.empty:
    print("Nenhum erro registrado.")
else:
    display(error_df.loc[:, ["IMG_ID", "label_file", "error"]].reset_index(drop=True))

Nenhum erro registrado.
